# Лекция 6. Нейросеть вместо таблицы: DQN

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IlyaChichkanov/Reinforcement-learning/blob/main/06-dqn/lecture/lecture.ipynb)

**Главный вопрос сегодня: как учить Q-функцию, когда состояний бесконечно много, — и зачем для этого нужны буфер опыта и целевая сеть?**

Карта пути. Q-learning (лекции 3 и 4) хранил таблицу $Q$: по числу на каждую пару «состояние, действие». Сегодня среда — CartPole из лекции 5, её состояние — четыре непрерывных числа. Таблица на сетке учится медленно: каждая клетка — отдельно. Заменим таблицу нейросетью — она учит похожие состояния вместе. Потом добавим два приёма, без которых сеть учится плохо: буфер опыта и целевую сеть. Вместе это **DQN** (Deep Q-Network) — алгоритм, который в 2015 году научился играм Atari прямо по пикселям экрана.

Лекция устроена как прошлые: раздел начинается с вопроса, на который мы **голосуем всей аудиторией**, и только потом проверяем ответ запуском. Вопросы и песочница — на странице-спутнике, откройте её на телефоне:

![страница лекции](attachment:lecture06_qr.png)

[ilyachichkanov.github.io/Reinforcement-learning/lecture06.html](https://ilyachichkanov.github.io/Reinforcement-learning/lecture06.html)

| | Раздел | Минут |
|---|---|---|
| 1 | 🔁 Таблица не учит соседей | 8 |
| 2 | Q-сеть: один шаг учит все похожие состояния | 14 |
| 3 | Буфер опыта: учимся на перемешанных переходах | 16 |
| 4 | Почему оценки разлетаются: смертельная триада | 14 |
| 5 | Целевая сеть: неподвижная мишень. Это и есть DQN | 24 |
| 6 | ⏱ Улучшения DQN | 6 |
| 7 | Итоги и где это работает | 8 |

Пометки: **🔁** — вспоминаем пройденное, **⏱** — резерв, если останется время. В каждом разделе первая строка — **главная идея**, а блоки **«Простыми словами»** пересказывают формулы без формул. Служебный код — в `dqn_plots.py` (таблица на сетке, карты, графики) и `cartpole_plots.py` из семинара 5 (рисунок тележки); читать его не нужно.

Ноутбук целиком считается около {{RUNTIME}}, вывод сохранён. Чтобы быстро пройти его самому, поставьте в следующей ячейке `QUICK = True`: один запуск вместо трёх и 200 эпизодов вместо 300 (числа в тексте тогда не совпадут).

In [ ]:
QUICK = False                                     # True — один запуск и 200 эпизодов вместо трёх и 300
SEEDS = range(1) if QUICK else range(3)
EPISODES = 200 if QUICK else 300

# Служебные модули. Локально они уже в репозитории, в Google Colab — скачиваются.
import pathlib, sys, urllib.request
REPO = "https://raw.githubusercontent.com/IlyaChichkanov/Reinforcement-learning/main/"
for name, folder in (("dqn_plots", "06-dqn/lecture"), ("cartpole_plots", "05-policy-gradient/seminar")):
    local = pathlib.Path("../..") / folder / f"{name}.py"
    if pathlib.Path(f"{name}.py").exists():
        continue
    if local.exists():
        sys.path.append(str(local.parent))
    else:
        urllib.request.urlretrieve(REPO + folder + f"/{name}.py", f"{name}.py")
import importlib.util, subprocess
for package in ("gymnasium", "torch"):
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
from dqn_plots import *                          # make_cartpole, GridTable, train_table, карты и графики
from cartpole_plots import draw_cartpole, animate_cartpole
start_time = time.time()

## 1. 🔁 Таблица не учит соседей

**Главная идея: клетка таблицы учится, только когда агент в неё попал, — похожие состояния учатся порознь.**

**Среда — CartPole** из лекции 5:

* *состояние* — четыре числа: положение тележки $x$, её скорость $\dot x$, угол шеста $\varphi$ и его угловая скорость $\dot\varphi$;
* *действия* — два: толкнуть тележку ← или →;
* *награда* — $+1$ за каждый шаг, пока шест стоит;
* *конец эпизода* — шест отклонился больше чем на 12°, тележка уехала за край или прошло 500 шагов.

In [ ]:
env = make_cartpole()
s, _ = env.reset(seed=0)
draw_cartpole(s, title="старт: шест почти вертикален, тележка в центре")
plt.show()

Чтобы завести таблицу $Q$, каждое из четырёх чисел режем на 6 отрезков: получается сетка из $6^4 = 1296$ клеток, в таблице $1296 \times 2$ чисел. Учим Q-learning из лекций 3 и 4 — функция `train_table` из `dqn_plots.py` делает ровно это:

$$
Q(s, a) \;\leftarrow\; Q(s, a) + \alpha\,\big(r + \gamma \max_{a'} Q(s', a') - Q(s, a)\big), \qquad \alpha = 0.1,\ \ \gamma = 0.99,
$$

действие ε-жадное, $\varepsilon$ падает от 1 до 0.05 за первые 10 000 шагов.

**Вопрос 1 (голосуем).** Таблица на сетке $6 \times 6 \times 6 \times 6$, {{EPISODES}} эпизодов. Сколько шагов шест будет стоять к концу обучения?

* **A.** около 500 — до самого конца эпизода;
* **B.** около 200;
* **C.** около 50–100;
* **D.** около 20, как у случайной стратегии.

In [ ]:
table = GridTable(bins=6)
results = [train_table(table, seed=k, episodes=EPISODES) for k in SEEDS]
lengths_table = np.array([lengths for _, lengths, _ in results])
untouched = np.mean([np.mean(visits == 0) for _, _, visits in results])
print("последние 50 эпизодов, шагов в среднем по запускам:", lengths_table[:, -50:].mean(axis=1).round(0))
print(f"клеток, которые не обновлялись ни разу: {untouched:.0%}")
plot_runs({"таблица 6⁴": lengths_table}, title="Q-learning на сетке")
plt.show()

Ответ **C — около 50–100 шагов**: к концу обучения шест стоит в среднем {{TABLE_LAST}} шагов из 500.

**Почему так выходит.** Клетка таблицы сдвигается, только когда агент в неё попал. Соседние клетки, где шест стоит почти так же, учатся порознь, и каждую агенту нужно посетить самому. {{UNTOUCHED}} клеток за всё обучение не обновились ни разу. Сетку можно нарезать умнее — например, только по углу и угловой скорости, — но это подсказка человека, который заранее знает, что важно. А в игре, где состояние — кадр $84 \times 84$ пикселя, одно и то же состояние почти никогда не повторяется, и таблица не выучит ничего.

> **Простыми словами.** Таблица — как ученик, который заучивает ответ на каждый экзаменационный билет отдельно: на билет, которого он не видел, он не ответит, даже если тот почти совпадает с выученным.

**Одной строкой:** таблица учит каждую клетку отдельно — там, где агент не был, она не знает ничего.

## 2. Q-сеть: один шаг учит все похожие состояния

**Главная идея: сеть считает $Q$ всех состояний одними и теми же весами, поэтому шаг обучения в одном состоянии сдвигает оценки всех похожих. Это называют обобщением.**

Заменим таблицу нейросетью: на вход — состояние (4 числа), на выход — два числа $Q(s, \leftarrow)$ и $Q(s, \rightarrow)$. Все веса сети вместе обозначим $\theta$, как в лекции 5. Действие выбираем так же, как по таблице: с наибольшим $Q$.

In [ ]:
def make_qnet(seed=0):
    # Q-сеть: состояние (4 числа) -> оценки двух действий (Q(s, ←), Q(s, →)); её веса — параметры θ
    torch.manual_seed(seed)
    return torch.nn.Sequential(torch.nn.Linear(4, 64), torch.nn.ReLU(),
                               torch.nn.Linear(64, 64), torch.nn.ReLU(),
                               torch.nn.Linear(64, 2))

qnet = make_qnet()
print("Q(старт, ←), Q(старт, →) до обучения:", qnet(torch.as_tensor(s)).detach().numpy().round(3))
print("параметров θ:", sum(p.numel() for p in qnet.parameters()))

Учим сеть, как таблицу, только вместо «сдвинуть клетку» — шаг градиента. Цель — та же, что у Q-learning:

$$
y \;=\; r + \gamma\,(1 - \text{конец})\, \max_{a'} Q_\theta(s', a'), \qquad \text{loss} \;=\; \big(y - Q_\theta(s, a)\big)^2 .
$$

Множитель $(1 - \text{конец})$ равен нулю, если на этом шаге эпизод кончился: будущих наград тогда нет. Цель $y$ считается **числом**: мы двигаем оценку $Q_\theta(s, a)$ к цели, а не цель к оценке, поэтому градиент через $y$ не пропускаем (`torch.no_grad()`). Формула над строкой кода:

* `td_targets` — цель $y$ для пачки переходов;
* `q_of` — $Q_\theta(s_t, a_t)$: сеть даёт таблицу «переход × 2 действия», и из строки $t$ берём столбец сыгранного действия $a_t$ — так же, как `log_probs_of` в лекции 5.

In [ ]:
def td_targets(net, r, s2, done, gamma=0.99):
    # y = r + γ (1 − конец) max_a′ Q(s′, a′); считается числом — без градиента
    with torch.no_grad():
        return r + gamma * (1 - done) * net(s2).max(dim=1).values

def q_of(qnet, s, a):
    # Q_θ(s_t, a_t): из строки t таблицы «переход × действие» — столбец a_t
    return qnet(s)[torch.arange(len(a)), a]

> **Простыми словами.** Сеть учится предсказывать, сколько ещё (с дисконтом) простоит шест, если сейчас толкнуть ← или →. Цель $y$ — «одна настоящая награда плюс моя же оценка следующего состояния».

Чем сеть лучше таблицы? Посмотрим, что делает **один** шаг обучения. Возьмём состояние ★: шест наклонён вправо на 0.05 рад и падает вправо со скоростью 0.5 рад/с, тележка в центре. Сдвинем $Q(★, \rightarrow)$ к цели одним шагом градиента и посмотрим на $Q(\cdot, \rightarrow)$ на плоскости «угол — угловая скорость» — той же, что на картах стратегии семинара 5.

**Вопрос 2 (голосуем).** Где изменится $Q(\cdot, \rightarrow)$ после одного шага в ★?

* **A.** только в клетке ★, как у таблицы;
* **B.** в небольшом пятне вокруг ★;
* **C.** почти на всей плоскости;
* **D.** нигде заметно: шаг слишком маленький.

In [ ]:
star = np.array([0.0, 0.0, 0.05, 0.5], dtype=np.float32)   # x, ẋ, φ, φ̇
far = np.array([0.0, 0.0, -0.2, -1.9], dtype=np.float32)    # дальний угол: шест падает влево
points, shape = plane()
shift, shift_star = footprint_net(make_qnet(), star, points, action=1, target=50.0, lr=0.01)
far_index = np.argmin(np.abs(points - far).sum(axis=1))
print(f"сдвиг Q(·, →) в ★: {shift_star:+.3f},   в дальнем углу (φ = −0.2, φ̇ = −1.9): {shift[far_index]:+.3f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
plot_footprint(footprint_table(table, star, points), shape, star, ax=axes[0], title="таблица 6⁴: одна клетка")
plot_footprint(shift / shift_star, shape, star, ax=axes[1], title="Q-сеть: вся плоскость")
plt.tight_layout(); plt.show()

Ответ **C — почти на всей плоскости**: $Q(\cdot, \rightarrow)$ сдвинулась на {{NET_STAR}} в самой ★ и на {{NET_FAR}} в дальнем углу, где шест падает в другую сторону. У таблицы сдвинулась одна клетка.

**Почему так выходит.** У таблицы у каждой клетки свои числа. У сети все состояния считаются одними и теми же весами $\theta$: шаг сдвигает веса, а через них — оценки всех состояний сразу, сильнее всего у похожих на ★. Поэтому сети не нужно побывать в каждом состоянии.

> **Простыми словами.** Сеть — ученик, который понял правило, а не заучил билеты: разобрал одну задачу — стал лучше решать и похожие.

У обобщения есть обратная сторона: шаг в ★ сдвинул оценки и там, где данных не было, — в том числе в состояниях $s'$, из которых берётся цель $y$. Это всплывёт в разделе 3.

**Одной строкой:** сеть обобщает — один шаг обучения сдвигает оценки всех похожих состояний.

## 3. Буфер опыта: учимся на перемешанных переходах

**Главная идея: вместо того чтобы учиться на каждом шаге подряд, сеть учится на случайных пачках из журнала всех прошлых переходов — так обучение быстрее. Но оценки $Q$ при этом разлетаются.**

Самый прямой способ учить Q-сеть — как таблицу: после каждого шага один шаг градиента на этом же переходе. Но соседние шаги эпизода почти одинаковы, и сеть всё время подстраивается под текущий кусок эпизода.

**Буфер опыта** (experience replay) — журнал всех переходов $(s, a, r, s', \text{конец})$ на 50 000 записей. Сеть учится на **случайной пачке** из 64 переходов журнала: в пачке перемешаны разные эпизоды и разные ситуации, и каждый переход используется много раз.

> **Простыми словами.** Повторяешь перемешанные карточки из всех тем, а не двести задач подряд на одну.

Функция `train_qnet` ниже — весь алгоритм обучения. Флаг `replay` меняет одну строку — какие переходы идут в шаг: последний или случайная пачка. Флаг `target` — тема раздела 5, пока он выключен. Чтобы видеть, во что верит сеть, перед каждым эпизодом запоминаем $\max_a Q_\theta(s_0, a)$ — оценку стартового состояния.

In [ ]:
class ReplayBuffer:
    # журнал переходов (s, a, r, s′, конец); когда он полон, новые записываются поверх самых старых
    def __init__(self, size=50_000):
        self.s, self.s2 = np.zeros((size, 4), np.float32), np.zeros((size, 4), np.float32)
        self.a, self.r, self.done = np.zeros(size, np.int64), np.zeros(size, np.float32), np.zeros(size, np.float32)
        self.size, self.n = size, 0

    def add(self, s, a, r, s2, done):
        i = self.n % self.size
        self.s[i], self.a[i], self.r[i], self.s2[i], self.done[i] = s, a, r, s2, done
        self.n += 1

    def batch(self, idx):
        return [torch.as_tensor(x[idx]) for x in (self.s, self.a, self.r, self.s2, self.done)]


def train_qnet(seed, replay, target, episodes=EPISODES, lr=1e-3, batch=64, tau=0.01, eps_steps=5000):
    # обучение Q-сети; возвращает длины эпизодов, оценки старта max_a Q(s_0, a) и обученную сеть
    rng, env = np.random.default_rng(seed), make_cartpole()
    qnet, target_net = make_qnet(seed), make_qnet(seed)          # целевая сеть (раздел 5) — копия основной
    opt = torch.optim.Adam(qnet.parameters(), lr=lr)
    buffer, step, lengths, q_start = ReplayBuffer(), 0, [], []
    for episode in range(episodes):
        s, _ = env.reset(seed=seed * 10_000 + episode)
        with torch.no_grad():
            q_start.append(qnet(torch.as_tensor(s)).max().item())
        for t in range(1, 501):
            eps = max(0.01, 1 - step / eps_steps)                    # ε: от 1 до 0.01 за eps_steps шагов
            if rng.random() < eps:
                a = int(rng.integers(2))
            else:
                with torch.no_grad():
                    a = int(qnet(torch.as_tensor(s)).argmax())
            s2, r, terminated, truncated, _ = env.step(a)
            buffer.add(s, a, r, s2, float(terminated)); step += 1
            if replay:                                               # случайная пачка из всего журнала
                idx = rng.integers(min(buffer.n, buffer.size), size=min(batch, buffer.n))
            else:                                                    # только последний переход
                idx = np.array([(buffer.n - 1) % buffer.size])
            S, A, R, S2, D = buffer.batch(idx)
            y = td_targets(target_net if target else qnet, R, S2, D)
            loss = ((y - q_of(qnet, S, A)) ** 2).mean()
            opt.zero_grad(); loss.backward(); opt.step()
            if target:                                               # θ̄ ← (1 − τ) θ̄ + τ θ
                with torch.no_grad():
                    for p_bar, p in zip(target_net.parameters(), qnet.parameters()):
                        p_bar.mul_(1 - tau).add_(tau * p)
            s = s2
            if terminated or truncated:
                break
        lengths.append(t)
    return np.array(lengths), np.array(q_start), qnet

**Вопрос 3 (голосуем).** Учим Q-сеть {{EPISODES}} эпизодов двумя способами: на каждом шаге подряд и на случайных пачках из буфера. Что даст буфер?

* **A.** шест будет стоять дольше, и оценки $Q$ останутся разумными;
* **B.** шест будет стоять дольше, но оценки $Q$ вырастут до тысяч;
* **C.** ничего не изменится;
* **D.** станет хуже: старые переходы собраны плохой стратегией.

Проверим: по {{SEEDS}} запуска на способ. Справа — оценка старта; пунктир — сколько она может быть самое большее: даже если шест простоит вечно, сумма наград $1 + 0.99 + 0.99^2 + \dots = 1 / (1 - 0.99) = 100$.

In [ ]:
runs = {}
for name, replay in (("по шагу подряд", False), ("буфер", True)):
    runs[name] = [train_qnet(seed, replay=replay, target=False) for seed in SEEDS]
lengths = {name: np.array([l for l, _, _ in rs]) for name, rs in runs.items()}
q_start = {name: np.array([q for _, q, _ in rs]) for name, rs in runs.items()}
for name in runs:
    print(f"{name:15s} шагов в среднем за эпизоды 100–{EPISODES}: {lengths[name][:, 100:].mean(axis=1).round(0)},"
          f"  наибольшая оценка старта: {np.abs(q_start[name]).max(axis=1).round(0)}")

fig, axes = plt.subplots(1, 2, figsize=(15, 4.2))
plot_runs({"таблица 6⁴": lengths_table, **lengths}, ax=axes[0], title="сколько стоит шест")
plot_q_start(q_start, ax=axes[1], title="оценка Q старта")
plt.tight_layout(); plt.show()

{{ANSWER3}}

**Почему оценки разлетаются.** Цель $y = r + \gamma \max Q_\theta(s')$ считает та же сеть, которую мы учим. Шаг поднимает $Q_\theta(s, a)$, а из-за обобщения (раздел 2) поднимается и $Q_\theta(s')$ соседнего, почти такого же состояния — то есть сама цель. Следующий шаг тянется к новой, более высокой цели, и так по кругу. У таблицы такого нет: клетки $s$ и $s'$ хранят разные числа. Почему без буфера оценки остались разумными, а с буфером — нет, разберём в разделе 4.

> **Простыми словами.** Это как догонять собственную тень: шагаешь к ней — она отодвигается.

**Одной строкой:** буфер перемешивает данные и ускоряет обучение, но оценки $Q$ при нём уходят далеко за возможный предел.

## 4. Почему оценки разлетаются: смертельная триада

**Главная идея: оценка может разлететься до бесконечности даже при нулевых наградах, если сойдутся три условия — общие веса, своя оценка в цели и данные не той стратегии, которую оцениваем. Буфер добавил третье.**

Самый маленький пример (Саттон и Барто, раздел 11.2). Два состояния: из 1 агент переходит в 2 с наградой 0, из 2 эпизод кончается тоже с наградой 0. Настоящие ценности — нули.

* **Оценка** — один общий вес $w$: $\hat v(1) = w$, $\hat v(2) = 2w$.
* **Данные** собрала другая стратегия: в журнале только переходы $1 \to 2$.
* **Обновление** — как в разделе 2: цель $0 + \gamma\, \hat v(2) = 2\gamma w$, шаг $w \leftarrow w + \alpha\,(2\gamma w - w) = \big(1 + \alpha(2\gamma - 1)\big)\, w$.

**Вопрос 4 (голосуем).** $w = 1$, $\gamma = 0.9$, $\alpha = 0.1$, 100 обновлений. Что будет с $w$?

* **A.** придёт к 0 — награды же нули;
* **B.** останется 1;
* **C.** вырастет до тысяч;
* **D.** будет колебаться около 0.

In [ ]:
def two_states(kind="все три условия", gamma=0.9, alpha=0.1, updates=100):
    w, w1, w2, history = 1.0, 1.0, 2.0, []
    for _ in range(updates):
        if kind == "все три условия":                    # общий вес, своя оценка в цели, только переход 1 → 2
            w += alpha * (gamma * 2 * w - w)
        elif kind == "свои данные (on-policy)":          # эпизод целиком: 1 → 2, потом 2 → конец
            w += alpha * (gamma * 2 * w - w)
            w += alpha * (0 - 2 * w) * 2
        elif kind == "цель — сумма наград G = 0":         # Монте-Карло вместо своей оценки
            w += alpha * (0 - w)
        elif kind == "таблица: свой вес у состояния":
            w1 += alpha * (gamma * w2 - w1)
        history.append(w1 if kind.startswith("таблица") else w)
    return np.array(history)

kinds = ["все три условия", "свои данные (on-policy)", "цель — сумма наград G = 0", "таблица: свой вес у состояния"]
curves = {kind: two_states(kind) for kind in kinds}
for kind, h in curves.items():
    print(f"{kind:32s} v̂(1) после 100 шагов: {h[-1]:.2g}")
plot_w_curves(curves, title="оценка v̂(1) по шагам обучения (правда: 0)")
plt.show()

Ответ **C — вырастет до тысяч**: за шаг $w$ умножается на $1 + 0.1 \cdot (1.8 - 1) = 1.08$, и $1.08^{100} \approx {{TRIAD_W}}$.

**Почему так выходит.** Шаг поднимает $\hat v(1)$ к цели $1.8\,w$, но поднять $\hat v(1)$ можно только через общий вес — а он поднимает и $\hat v(2) = 2w$, вдвое сильнее. Цель убегает быстрее оценки. Опустить $\hat v(2)$ мог бы переход из 2 в конец, но его в данных нет. Уберите любое из трёх условий — и пример сходится (кривые на картинке).

Эти условия называют **смертельной триадой**:

* **аппроксимация** — общие веса у многих состояний (сеть);
* **бутстрэппинг** — в цели стоит своя же оценка следующего состояния;
* **off-policy** — данные собраны не той стратегией, которую оцениваем.

**Теперь раздел 3.** У Q-сети первые два условия есть всегда. Без буфера сеть учится на только что сделанном шаге своей нынешней стратегии — данные почти свои, и оценки остались разумными. Буфер добавил третье условие: в журнале переходы старых стратегий, вплоть до случайной в начале обучения. Все три сошлись — оценки разлетелись. А стратегия-сеть REINFORCE в лекции 5 обучилась без приёмов, потому что в её весе стоят настоящие награды $G_t$, а не своя оценка, и учится она только на свежих эпизодах своей стратегии.

> **Простыми словами.** Оценка подпирает сама себя: подняли $\hat v(1)$ — поднялась цель, подняли ещё — цель ещё выше. Остановить это может только настоящая награда, а её в данных нет. В песочнице на странице лекции условия можно выключать по одному.

**Одной строкой:** общие веса + своя оценка в цели + чужие данные могут развалить обучение даже при нулевых наградах — буфер опыта добавляет Q-сети третье условие.

## 5. Целевая сеть: неподвижная мишень. Это и есть DQN

**Главная идея: цель $y$ считает не сама обучаемая сеть, а её копия, которая медленно догоняет оригинал; так цель перестаёт убегать, и с буфером опыта это даёт DQN.**

Буфер оставляем — без него сеть учится медленно. Чинить будем то, из-за чего цель убегает: её считает та же сеть, которую мы двигаем. **Целевая сеть** (target network) $Q_{\bar\theta}$ — копия основной, и цель считает она:

$$
y \;=\; r + \gamma\,(1 - \text{конец})\, \max_{a'} Q_{\bar\theta}(s', a'), \qquad \bar\theta \;\leftarrow\; 0.99\, \bar\theta + 0.01\, \theta \ \ \text{после каждого шага}.
$$

Веса копии $\bar\theta$ каждый шаг сдвигаются к весам основной сети на 1 % — копия догоняет оригинал примерно за сотню шагов. В статье DQN копию переставляли целиком раз в 10 000 шагов; смысл тот же. В `train_qnet` это флаг `target`: в `td_targets` идёт `target_net`, и после шага — строка $\bar\theta \leftarrow (1 - \tau)\,\bar\theta + \tau\,\theta$.

> **Простыми словами.** Мишень привязали к медленно ползущей копии: пока сеть делает шаг к цели, сама цель почти стоит на месте.

**Вопрос 5 (голосуем).** Четыре варианта: буфер да/нет × целевая сеть да/нет. Какой будет держать шест дольше всех и с разумными оценками $Q$?

* **A.** только буфер;
* **B.** только целевая сеть;
* **C.** буфер и целевая сеть вместе;
* **D.** без приёмов.

In [ ]:
for name, replay, target in (("целевая сеть", False, True), ("буфер + целевая сеть (DQN)", True, True)):
    runs[name] = [train_qnet(seed, replay=replay, target=target) for seed in SEEDS]
    lengths[name] = np.array([l for l, _, _ in runs[name]])
    q_start[name] = np.array([q for _, q, _ in runs[name]])

print(f"{'':28s} {'шагов, эп. 100–' + str(EPISODES):>20s} {'наибольшая оценка старта':>27s}")
for name in ("по шагу подряд", "буфер", "целевая сеть", "буфер + целевая сеть (DQN)"):
    print(f"{name:28s} {str(lengths[name][:, 100:].mean(axis=1).round(0)):>20s} "
          f"{str(np.abs(q_start[name]).max(axis=1).round(0)):>27s}")

fig, axes = plt.subplots(1, 2, figsize=(15, 4.2))
plot_runs({"буфер": lengths["буфер"], "целевая сеть": lengths["целевая сеть"],
           "DQN": lengths["буфер + целевая сеть (DQN)"]}, ax=axes[0], title="сколько стоит шест")
plot_q_start({"буфер": q_start["буфер"], "целевая сеть": q_start["целевая сеть"],
              "DQN": q_start["буфер + целевая сеть (DQN)"]}, ax=axes[1], title="оценка Q старта")
plt.tight_layout(); plt.show()

{{ANSWER5}}

> **Простыми словами.** Буфер даёт сети разнообразные примеры, целевая сеть — неподвижную мишень. По отдельности каждый приём не справляется: без буфера мало данных и они однообразные, без целевой сети мишень убегает.

Так держит шест сеть из лучшего запуска DQN: в каждом состоянии — действие с наибольшим $Q$, без случайных шагов.

In [ ]:
best = max(runs["буфер + целевая сеть (DQN)"], key=lambda run: run[0][-50:].mean())[2]
env = make_cartpole()
s, _ = env.reset(seed=0)
states = [s]
while True:
    with torch.no_grad():
        a = int(best(torch.as_tensor(s)).argmax())
    s, _, terminated, truncated, _ = env.step(a)
    states.append(s)
    if terminated or truncated:
        break
animate_cartpole(states, path="dqn_cartpole.gif", every=6)
print("шест простоял шагов:", len(states) - 1)

![обученная DQN держит шест](attachment:dqn_cartpole.gif)

**Весь DQN в шести шагах** — то, что делает `train_qnet(seed, replay=True, target=True)`:

1. в состоянии $s$ выбрать действие ε-жадно по $Q_\theta$;
2. сделать шаг, положить $(s, a, r, s', \text{конец})$ в буфер;
3. взять из буфера случайную пачку;
4. посчитать цель $y = r + \gamma\,(1 - \text{конец}) \max_{a'} Q_{\bar\theta}(s', a')$ целевой сетью;
5. шаг градиента по $\big(y - Q_\theta(s, a)\big)^2$;
6. подтянуть копию: $\bar\theta \leftarrow 0.99\,\bar\theta + 0.01\,\theta$.

**Одной строкой:** DQN = Q-сеть + буфер опыта (разнообразные данные) + целевая сеть (неподвижная мишень).

## 6. ⏱ Улучшения DQN

**Главная идея: все улучшения DQN — о том, чтобы меньше опираться на завышенные или устаревшие собственные оценки.**

Оценка старта у DQN — до {{DQN_Q}} при пределе 100: даже с целевой сетью $\max$ в цели слегка завышает. Это смещение максимума из лекции 4, и первое улучшение — про него.

| улучшение | что меняет | зачем |
|---|---|---|
| Double DQN | действие в цели выбирает основная сеть, а оценивает целевая: $y = r + \gamma\, Q_{\bar\theta}\big(s', \arg\max_{a'} Q_\theta(s', a')\big)$ | смещение максимума (лекция 4): максимум шумных оценок завышен; вторая оценка уже есть — целевая сеть |
| Dueling DQN | сеть отдельно оценивает, насколько хорошо состояние, $V(s)$, и насколько лучше каждое действие, $A(s, a)$ | там, где действие почти не важно, $V$ учится по всем действиям сразу |
| Prioritized replay | из буфера чаще берутся переходы с большой ошибкой $\lvert y - Q \rvert$ | учимся там, где сеть ошибается сильнее |
| n-step | в цели $n$ настоящих наград, потом оценка | меньше опоры на свою оценку |
| Rainbow | всё вместе и ещё два приёма | на Atari лучше каждого улучшения по отдельности |

**Одной строкой:** улучшения DQN уменьшают завышение и опору на собственные оценки.

## 7. Итоги и где это работает

1. **Таблица учит каждую клетку отдельно:** на CartPole сетка из 1296 клеток держит шест {{TABLE_LAST}} шагов.
2. **Q-сеть обобщает:** один шаг обучения сдвигает $Q$ почти на всей плоскости состояний, а не в одной клетке.
3. **Буфер опыта** перемешивает данные и ускоряет обучение, но вместе с общими весами и своей оценкой в цели даёт **смертельную триаду**: оценка старта выросла до {{REPLAY_Q}} при пределе 100.
4. **Целевая сеть** — медленная копия основной — держит цель на месте. **DQN = Q-сеть + буфер + целевая сеть:** {{SUMMARY5}}
5. **Почему сеть лучше таблицы:** обобщает похожие состояния, не требует ручной сетки, работает с картинками. Плата — неустойчивость, её лечат буфер и целевая сеть.

**Где это работает в жизни.**

* **Игры Atari.** DQN (Mnih и др., 2015): $Q$-функция — свёрточная сеть по четырём последним кадрам экрана $84 \times 84$; одна и та же программа с одними настройками научилась 49 играм, во многих — на уровне человека.
* **Нарды.** TD-Gammon (Тезауро, 1992–1995) — предшественник DQN: сеть оценивала позицию и училась на партиях против себя.
* **Рекомендации.** DQN и его потомки выбирают, что показать пользователю, когда вариантов немного, а состояние — длинный вектор признаков.

> **Простыми словами — вся лекция.** Таблица заучивает билеты, сеть понимает правило. Но сеть, которая учится по собственным оценкам на старых данных, может «убедить сама себя» в бессмыслице. DQN не даёт ей этого сделать: учит на перемешанных примерах и сверяется с медленной копией себя.

Словарь лекции: **Q-сеть, обобщение, цель $y$, буфер опыта (experience replay), смертельная триада (аппроксимация, бутстрэппинг, off-policy), целевая сеть (target network), DQN, Double DQN**.

## Литература

* Mnih V. и др. *Human-level control through deep reinforcement learning*. Nature, 2015 — DQN на Atari.
* Саттон Р., Барто Э. *Обучение с подкреплением*, 2-е изд., глава 11 (пример с $w$ и $2w$ — раздел 11.2, смертельная триада — раздел 11.3). [incompleteideas.net/book/the-book-2nd.html](http://incompleteideas.net/book/the-book-2nd.html)
* van Hasselt H., Guez A., Silver D. *Deep reinforcement learning with double Q-learning*. AAAI, 2016 — Double DQN.
* Hessel M. и др. *Rainbow: combining improvements in deep reinforcement learning*. AAAI, 2018.
* [Учебник по машинному обучению ШАД, глава «Обучение с подкреплением»](https://education.yandex.ru/handbook/ml/article/obuchenie-s-podkrepleniem).

## На семинаре и дома

* **Домашка блока 3** — дедлайн объявит преподаватель.

In [ ]:
print(f"ноутбук посчитался за {(time.time() - start_time) / 60:.0f} мин")